In [1]:
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorboard.plugins.hparams import api as hp

In [2]:
df = pd.read_excel('..\\..\\Datasets\\NewDataset\\BlackLineHospitalLungCancerDataset.xlsx', header=0) 

X = df.drop(['Severity'], axis=1)
y = df['Severity']

In [3]:
nums = []
TuningTrials = 3

def run(hparams, num, i): 
        hp.hparams(hparams)
        accuracy = train_test_model(hparams)
        #print(accuracy)
        if (i == 0):
            nums.append(accuracy)
        else: 
            nums[num] += accuracy
        if (i == TuningTrials-1):
            average = nums[num]/TuningTrials
            nums[num] = average
            #print(average)
            

In [4]:
def train_test_model(hparams):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(4, activation=tf.nn.softmax)
    ])
    model.compile(
        optimizer=hparams[HP_OPTIMIZER],
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    model.fit(
        X_train,
        y_train,
        epochs=hparams[HP_EPOCHS],
        verbose = 0,
        
    ) 
    _, accuracy = model.evaluate(X_test, y_test)
    return accuracy

In [5]:
Permutations = []
for i in range(TuningTrials):
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = .2, shuffle=True)
    datasetX = tf.convert_to_tensor(X_train)
    datasetY = tf.convert_to_tensor(y_train)
    #I manually narrowed down these domains while I tested for specific hyperparameters due to computational and time constraints. 
    # I didn't have time to grid search every realistic combination of these values
    HP_NUM_UNITS = hp.HParam('num_units', hp.Discrete(list(range(68, 75, 1)))) 
    HP_DROPOUT = hp.HParam('dropout', hp.RealInterval(0.0, 0.0)) #hp.RealInterval(0.0, 0.1)
    HP_OPTIMIZER = hp.HParam('optimizer', hp.Discrete(['Adam'])) #, 'RMSprop', 'Sgd'
    HP_EPOCHS = hp.HParam('epochs', hp.Discrete(list(range(50, 54, 1)))) 
    METRIC_ACCURACY = 'accuracy'
    session_num = 0
    for num_units in HP_NUM_UNITS.domain.values:
        for dropout_rate in (HP_DROPOUT.domain.min_value, HP_DROPOUT.domain.max_value):
            for optimizer in HP_OPTIMIZER.domain.values:
                for epoch in HP_EPOCHS.domain.values:
                    hparams = {
                        HP_NUM_UNITS: num_units,
                        HP_DROPOUT: dropout_rate,
                        HP_OPTIMIZER: optimizer,
                        HP_EPOCHS: epoch,
                    }
                    # run_name = "run-%d" %  i + "-" + str(session_num)
                    # print('--- Starting trial: %s' % run_name)
                    # #print({h.name: hparams[h] for h in hparams})
                    Permutations.append({h.name: hparams[h] for h in hparams})
                    run(hparams=hparams, num=session_num, i=i) 
                    session_num += 1

for trial in nums:
    trial = trial / TuningTrials


Best = max(nums)
BestCombination = Permutations[nums.index(Best)]
print(f"Best trial: {Best}")
print(f"Best combination: {BestCombination}")

optimalUnits = BestCombination['num_units']
optimalDropout = BestCombination['dropout']
optimalOptimizer = BestCombination['optimizer']
optimalEpochs = BestCombination['epochs']



10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9113 - loss: 0.2930  
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9625 - loss: 0.1670  
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9625 - loss: 0.0939  
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9556 - loss: 0.1462  
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9522 - loss: 0.1391  
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9659 - loss: 0.1287  
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9727 - loss: 0.1107  
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9488 - loss: 0.1778  
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9454 - loss: 0.1270  
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9590 - loss: 0.1709  
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9420 - loss: 0.1781  
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9556 - loss: 0.1230  
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9693 - loss: 0.1562  
10/10 ━━━━━━━━━━━━━━━━━━━

In [10]:

def train_test_tuned_model():
    model = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(optimalUnits, activation=tf.nn.relu),
        tf.keras.layers.Dropout(optimalDropout),
        tf.keras.layers.Dense(optimalUnits, activation=tf.nn.relu),
        tf.keras.layers.Dropout(optimalDropout),
        tf.keras.layers.Dense(optimalUnits, activation=tf.nn.relu),
        tf.keras.layers.Dropout(optimalDropout),
        tf.keras.layers.Dense(optimalUnits, activation=tf.nn.relu),
        tf.keras.layers.Dropout(optimalDropout),
        tf.keras.layers.Dense(optimalUnits, activation=tf.nn.relu),
        tf.keras.layers.Dropout(optimalDropout),
        tf.keras.layers.Dense(4, activation=tf.nn.softmax)
    ])
    model.compile(
        optimizer= optimalOptimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    model.fit(
        X_train,
        y_train,
        epochs=optimalEpochs,
        verbose=0,
        
    )
    _, accuracy = model.evaluate(X_test, y_test)
    return accuracy


sum = 0
TestTrials = 10

for i in range(TestTrials):
    run_name = (i+1)
    print('--- Starting trial: %s' % run_name)
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = .2, shuffle=True)
    datasetX = tf.convert_to_tensor(X_train)
    datasetY = tf.convert_to_tensor(y_train)
    accuracy = train_test_tuned_model()
    sum += accuracy
    print(f"Trial {i+1} accuracy: {accuracy}\n")
    
average = sum / TestTrials

print("Average Accuracy: ", average)


--- Starting trial: 1
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9659 - loss: 0.1158  
Trial 1 accuracy: 0.9658703207969666

--- Starting trial: 2
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9761 - loss: 0.1169  
Trial 2 accuracy: 0.9761092066764832

--- Starting trial: 3
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9693 - loss: 0.0751  
Trial 3 accuracy: 0.9692832827568054

--- Starting trial: 4
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9761 - loss: 0.0714  
Trial 4 accuracy: 0.9761092066764832

--- Starting trial: 5
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9590 - loss: 0.1704  
Trial 5 accuracy: 0.9590443968772888

--- Starting trial: 6
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9863 - loss: 0.0581  
Trial 6 accuracy: 0.9863481521606445

--- Starting trial: 7
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9625 - loss: 0.1245  
Trial 7 accuracy: 0.9624573588371277

--- Starting trial: 8
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms